In [ ]:
import os
import torch
import clip
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pathlib import Path
from tqdm import tqdm

base_path = Path("/Users/aurora/Desktop/Python_AI and Unreal Workshop/Web Scraping")
pic_dir = base_path / "pictures"
film_dir = base_path / "films"

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def get_data(folder):
    feats = []
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:100]
    for f in tqdm(files, desc=f"Processing {folder.name}"):
        try:
            with Image.open(folder / f) as img:
                img_tensor = preprocess(img.convert("RGB")).unsqueeze(0).to(device)
                with torch.no_grad():
                    vi = model.encode_image(img_tensor)
                    vi /= vi.norm(dim=-1, keepdim=True)
                    feats.append(vi.cpu().numpy().flatten())
        except:
            continue
    return np.array(feats)

print("--- Start extracting architecture features ---")
arch_vecs = get_data(pic_dir)

print("--- Start extracting film features ---")
film_vecs = get_data(film_dir)

if len(arch_vecs) > 0 and len(film_vecs) > 0:
    print("--- Running dimensionality reduction ---")
    all_vecs = np.vstack([arch_vecs, film_vecs])
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_vecs)-1))
    reduced = tsne.fit_transform(all_vecs)

    plt.figure(figsize=(10, 7))
    n_arch = len(arch_vecs)

    plt.scatter(reduced[:n_arch, 0], reduced[:n_arch, 1], c='blue', label='Architecture', alpha=0.5)
    plt.scatter(reduced[n_arch:, 0], reduced[n_arch:, 1], c='red', marker='x', label='Film', alpha=0.5)

    plt.legend()
    plt.title("Gothic Style Consistency Analysis")

    out = os.path.expanduser("~/Desktop/Gothic_Map.png")
    plt.savefig(out)
    print(f"--- Done. Saved to: {out} ---")

    plt.show()
else:
    print("Feature extraction failed. Check if folders contain images.")
